Key Points & Enhancements
✅ What this script does
Queries only the latest version of each file (IsLatest = true) – avoids duplicates.

Downloads the binary data using the VersionData URL (requires session ID).

Re‑uploads to target org as a new ContentVersion linked to the mapped parent record.

🔁 Linking to Parent Records
The critical piece is PARENT_ID_MAP. You typically generate this by:

First migrating standard/custom objects (Accounts, Opportunities, etc.) with an External ID field.

Querying source and target orgs to build a dictionary: source_id -> target_id.

If you haven't migrated parent records yet, you can extend the script to match by External ID dynamically:

In [ ]:
# Example: find target account by external ID
ext_id = 'EXT_12345'  # from source record's external ID field
target_account = sf_target.query(
    f"SELECT Id FROM Account WHERE External_Id__c = '{ext_id}'"
)
if target_account['totalSize'] > 0:
    target_parent_id = target_account['records'][0]['Id']

Handling Attachments (Legacy)
If you need to migrate Attachment objects instead:

In [ ]:
# Query Attachment
att = sf_source.query("SELECT Id, Name, Body, ParentId FROM Attachment WHERE ...")

# Upload to target
sf_target.Attachment.create({
    'Name': att['Name'],
    'ParentId': target_parent_id,
    'Body': att['Body']  # already base64
})

Performance Tips
Bulk API – For >10k files, replace create() with sf_target.bulk.ContentVersion.insert().

Parallel downloads – Use concurrent.futures to download/upload in batches of 5–10 (respect API limits).

Resume capability – Keep a log of successfully migrated ContentDocumentId to avoid re‑migrating.

🛡️ Error Handling & Logging
Add these for production:

Retry logic (exponential backoff) on timeout/rate limits.

Detailed CSV log of successes/failures.

Verify file size doesn't exceed 2GB (ContentVersion limit).

Here's a production‑ready Python module that handles automatic parent mapping via External IDs and uses high‑volume techniques (Bulk API for queries + concurrent uploads). It also includes resumability and proper error handling.

In [ ]:
"""
salesforce_file_migrator.py

Kernel: base

Salesforce File Migrator (ContentVersion)
- Maps parent records using External IDs
- Downloads/Uploads files with concurrency & retries
- Supports resuming from previous checkpoint
"""

import base64
import csv
import logging
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Optional, List, Set

import requests
from simple_salesforce import Salesforce
# from simple_salesforce import SalesforceBulk


In [2]:
# ---------- Configuration ----------
LOG_LEVEL = logging.INFO
MAX_WORKERS = 5               # Concurrent uploads (respect API limits)
MAX_RETRIES = 3
RETRY_DELAY = 2               # seconds
CHECKPOINT_FILE = 'migrated_files.txt'   # stores migrated ContentDocument IDs
BATCH_SIZE = 200              # for query pagination
# ----------------------------------

logging.basicConfig(level=LOG_LEVEL, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
class SalesforceFileMigrator:
    def __init__(self, source_config: dict, target_config: dict):
        """
        source_config / target_config = {
            'username': '...',
            'password': '...',
            'security_token': '...' (or None),
            'domain': 'login' (or 'test')
        }
        """
        self.source = self._connect(source_config)
        self.target = self._connect(target_config)

        # For Bulk API queries (large result sets)
        # self.source_bulk = SalesforceBulk(
        self.source_bulk = Salesforce(
            username=source_config['username'],
            password=source_config['password'],
            security_token=source_config.get('security_token'),
            instance_url=source_config.get('instance_url'),
            domain=source_config.get('domain', 'login'),
            version="59.0"
        )

        # Track migrated files
        self.migrated_doc_ids = self._load_checkpoint()

    def _connect(self, config: dict) -> Salesforce:
        return Salesforce(
            username=config['username'],
            password=config['password'],
            security_token=config.get('security_token'),
            instance_url=config.get('instance_url'),
            domain=config.get('domain', 'login'),
            version="59.0"
        )

    def _load_checkpoint(self) -> Set[str]:
        """Load already migrated ContentDocument IDs from file."""
        if os.path.exists(CHECKPOINT_FILE):
            with open(CHECKPOINT_FILE, 'r') as f:
                return set(line.strip() for line in f)
        return set()

    def _save_checkpoint(self, content_doc_id: str):
        """Append migrated ContentDocument ID to checkpoint file."""
        with open(CHECKPOINT_FILE, 'a') as f:
            f.write(f"{content_doc_id}\n")
        self.migrated_doc_ids.add(content_doc_id)

    def build_parent_map(self, object_name: str, external_id_field: str, source_ids: List[str]) -> Dict[str, str]:
        """
        Build mapping: source_record_id -> target_record_id using External ID.
        :param object_name: e.g., 'Account'
        :param external_id_field: e.g., 'External_Id__c'
        :param source_ids: list of source record IDs (ParentId values from ContentVersion)
        :return: dict { source_id: target_id }
        """
        if not source_ids:
            return {}

        # 1. Query source records to get external ID values
        #    Use a single query with IN clause (max 2000 IDs)
        id_chunks = [source_ids[i:i + 2000] for i in range(0, len(source_ids), 2000)]
        source_ext_id_map = {}  # source_id -> external_id_value

        for chunk in id_chunks:
            ids_str = "','".join(chunk)
            query = f"SELECT Id, {external_id_field} FROM {object_name} WHERE Id IN ('{ids_str}')"
            results = self.source.query_all(query)
            for rec in results['records']:
                ext_val = rec.get(external_id_field)
                if ext_val:
                    source_ext_id_map[rec['Id']] = ext_val
                else:
                    logger.warning(f"Source {object_name} {rec['Id']} has no external ID, skipping")

        if not source_ext_id_map:
            logger.warning("No valid external IDs found in source")
            return {}

        # 2. Query target org to find matching external IDs
        target_id_by_ext = {}
        ext_values = list(source_ext_id_map.values())
        ext_chunks = [ext_values[i:i + 2000] for i in range(0, len(ext_values), 2000)]

        for chunk in ext_chunks:
            ext_str = "','".join(chunk)
            query = f"SELECT Id, {external_id_field} FROM {object_name} WHERE {external_id_field} IN ('{ext_str}')"
            results = self.target.query_all(query)
            for rec in results['records']:
                target_id_by_ext[rec[external_id_field]] = rec['Id']

        # 3. Build final mapping
        parent_map = {}
        for src_id, ext_val in source_ext_id_map.items():
            target_id = target_id_by_ext.get(ext_val)
            if target_id:
                parent_map[src_id] = target_id
            else:
                logger.warning(f"No target record found for external ID {ext_val} (source ID {src_id})")

        logger.info(f"Built parent mapping: {len(parent_map)} of {len(source_ids)} records mapped")
        return parent_map

    def get_files_to_migrate(self, parent_map: Dict[str, str], query_filter: str = "") -> List[dict]:
        """
        Query ContentVersion (latest version only) where parent is in parent_map.
        Returns list of dicts with necessary fields.
        """
        parent_ids = list(parent_map.keys())
        if not parent_ids:
            return []

        # Build SOQL with parent filter
        parent_id_str = "','".join(parent_ids)
        soql = f"""
            SELECT Id, ContentDocumentId, Title, PathOnClient, FileExtension,
                   VersionData, FirstPublishLocationId
            FROM ContentVersion
            WHERE IsLatest = true
              AND FirstPublishLocationId IN ('{parent_id_str}')
              {query_filter}
        """
        logger.info("Querying source for files...")
        # Use Bulk API for large volumes
        job = self.source_bulk.create_query_job('ContentVersion', contentType='CSV')
        batch = self.source_bulk.query(job, soql)
        self.source_bulk.wait_for_job(job)
        results = self.source_bulk.get_all_results_for_query_batch(job, batch)

        files = []
        for row in results:
            # row is a dict from CSV
            files.append({
                'id': row['Id'],
                'content_document_id': row['ContentDocumentId'],
                'title': row['Title'],
                'path_on_client': row.get('PathOnClient', row['Title']),
                'file_extension': row.get('FileExtension', ''),
                'version_data_url': row['VersionData'],
                'source_parent_id': row['FirstPublishLocationId']
            })
        logger.info(f"Found {len(files)} files to migrate")
        return files

    def download_file(self, version_data_url: str, session_id: str) -> Optional[bytes]:
        """Download binary file from source org."""
        headers = {'Authorization': f'Bearer {session_id}'}
        for attempt in range(MAX_RETRIES):
            try:
                resp = requests.get(version_data_url, headers=headers, timeout=60)
                if resp.status_code == 200:
                    return resp.content
                else:
                    logger.warning(f"Download attempt {attempt+1} failed: {resp.status_code}")
            except Exception as e:
                logger.warning(f"Download error: {e}")
            time.sleep(RETRY_DELAY)
        return None

    def upload_file(self, file_info: dict, target_parent_id: str) -> bool:
        """Upload a single ContentVersion to target org."""
        # Download binary
        binary = self.download_file(file_info['version_data_url'], self.source.session_id)
        if binary is None:
            logger.error(f"Failed to download file {file_info['title']} (ID {file_info['id']})")
            return False

        # Prepare ContentVersion object
        version_data_b64 = base64.b64encode(binary).decode('utf-8')
        new_cv = {
            'Title': file_info['title'],
            'PathOnClient': file_info['path_on_client'],
            'FirstPublishLocationId': target_parent_id,
            'VersionData': version_data_b64,
            'ContentLocation': 'S'
        }
        if file_info['file_extension']:
            new_cv['FileExtension'] = file_info['file_extension']

        # Upload via REST (Bulk API not ideal for binary base64 fields)
        try:
            result = self.target.ContentVersion.create(new_cv)
            logger.info(f"✅ Uploaded {file_info['title']} -> {result['id']}")
            return True
        except Exception as e:
            logger.error(f"Upload failed for {file_info['title']}: {e}")
            return False

    def migrate(self, object_name: str, external_id_field: str, query_filter: str = ""):
        """
        Main migration workflow:
        1. Find all source parent IDs that have files.
        2. Build parent mapping using external IDs.
        3. Migrate files with concurrency & checkpoint.
        """
        # Step 1: Get distinct parent IDs from ContentVersion
        logger.info("Collecting source parent IDs with files...")
        # parent_query = """
        #     SELECT DISTINCT FirstPublishLocationId
        #     FROM ContentVersion
        #     WHERE IsLatest = true
        #       AND FirstPublishLocationId != NULL
        # """
        parent_query = """
            SELECT FirstPublishLocationId
            FROM ContentVersion
            WHERE IsLatest = true
              AND FirstPublishLocationId != NULL
        """
        results = self.source.query_all(parent_query)
        source_parent_ids = [row['FirstPublishLocationId'] for row in results['records']]
        logger.info(f"Found {len(source_parent_ids)} unique parent records with files")

        # Step 2: Build parent mapping
        parent_map = self.build_parent_map(object_name, external_id_field, source_parent_ids)
        if not parent_map:
            logger.error("No parent mapping built. Aborting.")
            return

        # Step 3: Get files to migrate (filtered by mapped parents)
        files = self.get_files_to_migrate(parent_map, query_filter)

        # Filter out already migrated files (checkpoint by ContentDocumentId)
        pending_files = [f for f in files if f['content_document_id'] not in self.migrated_doc_ids]
        logger.info(f"{len(pending_files)} files pending after checkpoint")

        if not pending_files:
            logger.info("No new files to migrate.")
            return

        # # Step 4: Concurrent uploads
        # success_count = 0
        # with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        #     future_to_file = {}
        #     for f in pending_files:
        #         target_parent = parent_map[f['source_parent_id']]
        #         future = executor.submit(self.upload_file, f, target_parent)
        #         future_to_file[future] = f

        #     for future in as_completed(future_to_file):
        #         f = future_to_file[future]
        #         try:
        #             if future.result():
        #                 self._save_checkpoint(f['content_document_id'])
        #                 success_count += 1
        #         except Exception as e:
        #             logger.error(f"Exception for file {f['title']}: {e}")

        logger.info(f"Migration complete. Success: {success_count}/{len(pending_files)}")


In [4]:
# ---------- Usage Example ----------
# if __name__ == '__main__':
source_cfg = {
    'username': 'rato.li@rhombusadvisory.com.au.hubuat',
    'password': 'l1qt150_on_BF26',
    'security_token': 'mlMvNgFbW6UiniZoBLt4M42zp',
    'instance_url': 'https://ioofadviserhub--hubuat.sandbox.lightning.force.com',
    'domain': 'test'
}
target_cfg = {
    'username': 'rato.li@rhombusenterprises.com.au.rhuat',
    'password': 'l1qt100_on_BF26',
    'security_token': 'dqtmCCM25oNMtD72F74VjfWpl',
    'instance_url': 'https://test.salesforce.com',
    'domain': 'test'
}

migrator = SalesforceFileMigrator(source_cfg, target_cfg)


In [5]:
source_desc = migrator.source.describe()

prefix_map = {}

for obj in source_desc['sobjects']:
    if 'keyPrefix' in obj and obj['keyPrefix']:
        prefix_map[obj['keyPrefix']] = obj['name']

def resolve_object(sf_id):
    return prefix_map.get(sf_id[:3])


In [8]:
prefix_map['005']

'User'

In [15]:
migrator.migrate(
    object_name='Account',           # Parent object type
    external_id_field='External_Id__c',  # Unique external ID field on parent
    query_filter="AND FileExtension IN ('pdf', 'docx')"   # optional filter
)

2026-05-28 23:40:56,602 - INFO - Collecting source parent IDs with files...
2026-05-28 23:40:56,879 - INFO - Found 532 unique parent records with files


SalesforceMalformedRequest: Malformed request https://ioofadviserhub--hubuat.sandbox.my.salesforce.com/services/data/v59.0/query/?q=SELECT+Id%2C+External_Id__c+FROM+Account+WHERE+Id+IN+%28%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000AgiMpQAJ%27%2C%270056F00000AQVHEQA5%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DnqAxQAJ%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000CZ1Z7QAL%27%2C%270586F000000q2hwQAA%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270586F000000q2hwQAA%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000DYE2TQAX%27%2C%270056F00000COxAlQAL%27%2C%2